In [2]:
import gradio as gr
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import torch.nn.functional as F

print("Gradio version:", gr.__version__)
print("Ready to build the app!")

Gradio version: 6.19.0
Ready to build the app!


In [3]:
# Define class names
classes = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

# Load model
model = models.efficientnet_b0(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(1280, 6)
)
model.load_state_dict(torch.load("clearbin_best.pth"))
model.eval()

# Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

print("Model loaded!")
print("Classes:", classes)

Model loaded!
Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


In [4]:
# Prediction function
def predict(image):
    # Convert to PIL if needed
    img = Image.fromarray(image).convert("RGB")
    
    # Apply transforms
    img_tensor = transform(img).unsqueeze(0)  # Add batch dimension
    
    # Get prediction
    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = F.softmax(outputs, dim=1)[0]
    
    # Return confidence scores for each class
    return {classes[i]: float(probabilities[i]) for i in range(len(classes))}

# Build Gradio app
app = gr.Interface(
    fn=predict,
    inputs=gr.Image(),
    outputs=gr.Label(num_top_classes=6),
    title="ClearBin ♻️ — AI Waste Classifier",
    description="Upload a photo of waste and ClearBin will tell you which bin it belongs to!",
    examples=[["test.jpg"]],
    theme=gr.themes.Soft()
)

app.launch()

C:\clearbin\venv\Lib\site-packages\gradio\interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


C:\clearbin\venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\clearbin\venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Created dataset file at: .gradio\flagged\dataset1.csv


C:\clearbin\venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
